# Exam: Time Series Visualization with Bokeh

This exam tests your ability to visualize time series data using the Bokeh library.
You will be working with the "Daily Minimum Temperatures in Melbourne" dataset.
For each question, provide the Python code using Bokeh to generate the requested visualization.

**Dataset:** "daily-minimum-temperatures-in-melbourne.csv"

```python
import pandas as pd
from bokeh.plotting import figure, show
from bokeh.io import output_notebook
from bokeh.models import (
    ColumnDataSource,
    HoverTool,
    DatetimeTickFormatter,
    NumeralTickFormatter,
)
from bokeh.layouts import row, column
from bokeh.transform import factor_cmap

output_notebook()  # Enable Bokeh output in Jupyter Notebook

# Load the Dataset
df = pd.read_csv("daily-minimum-temperatures-in-melbourne.csv")

# Rename columns for clarity
df.columns = ['Date', 'Temperature']

# Convert the 'Date' column to datetime format
df['Date'] = pd.to_datetime(df['Date'])

# Remove '?' from the 'Temperature' column and convert to numeric
df['Temperature'] = df['Temperature'].astype(str).str.replace('?', '', regex=False)
df['Temperature'] = pd.to_numeric(df['Temperature'])

Question 1: Basic Time Series Line Plot
1.  Create a basic line plot showing the daily minimum temperature over time.

    * Use the 'Date' column on the x-axis and the 'Temperature' column on the y-axis.
    * Set the plot title to "Daily Minimum Temperatures".
    * Label the x-axis as "Date" and the y-axis as "Temperature (°C)".
    * Add tooltips to display the date and temperature when hovering over the line.
    * Enable pan, wheel zoom, and reset tools.


In [12]:
import pandas as pd
from bokeh.plotting import figure, show
from bokeh.io import output_notebook
from bokeh.models import (
    ColumnDataSource,
    HoverTool,
    DatetimeTickFormatter,
    NumeralTickFormatter,
)
from bokeh.layouts import row, column
from bokeh.transform import factor_cmap

output_notebook()

df = pd.read_csv("../datasets/daily-minimum-temperatures-in-melbourne.csv")
df.columns = ['Date', 'Temperature']
df['Date'] = pd.to_datetime(df['Date'])
df['Temperature'] = df['Temperature'].astype(str).str.replace('?', '', regex=False)
df['Temperature'] = pd.to_numeric(df['Temperature'])

Loading BokehJS ...

In [16]:
source = ColumnDataSource(df)

hover = HoverTool(tooltips=[
    ("Date", "@Date{%F}"),
    ("Temperature", "@Temperature °C")
], formatters={"@Date": "datetime"})

p = figure(
    title="Daily Minimum Temperatures",
    x_axis_label="Date",
    y_axis_label="Temperature (°C)",
    x_axis_type="datetime",
    tools="pan,wheel_zoom,reset",
    width=900, height=400
)
p.add_tools(hover)
p.line("Date", "Temperature", source=source, line_width=1, color="steelblue")
show(p)

Question 2: Rolling Average
2.  Calculate the 30-day rolling average of the daily minimum temperature and plot it
    alongside the original temperature data.

    * Create a new column 'Rolling_Avg' in the DataFrame containing the 30-day rolling average.
    * Plot both the original 'Temperature' and the 'Rolling_Avg' on the same plot.
    * Use different colors and line styles to distinguish between the two.
    * Add a legend to the plot to label the lines.
    * Add tooltips to display the date, original temperature, and rolling average.

In [17]:
df['Rolling_Avg'] = df['Temperature'].rolling(window=30).mean()
source2 = ColumnDataSource(df)

hover2 = HoverTool(tooltips=[
    ("Date", "@Date{%F}"),
    ("Temperature", "@Temperature °C"),
    ("Rolling Avg", "@Rolling_Avg{0.2f} °C")
], formatters={"@Date": "datetime"})

p2 = figure(
    title="Daily Minimum Temperatures with 30-Day Rolling Average",
    x_axis_label="Date",
    y_axis_label="Temperature (°C)",
    x_axis_type="datetime",
    tools="pan,wheel_zoom,reset",
    width=900, height=400
)
p2.add_tools(hover2)

p2.line("Date", "Temperature", source=source2,
        line_width=1, color="steelblue", alpha=0.5, legend_label="Daily Temperature")
p2.line("Date", "Rolling_Avg", source=source2,
        line_width=2.5, color="firebrick", line_dash="dashed", legend_label="30-Day Rolling Avg")

p2.legend.location = "top_left"
p2.legend.click_policy = "hide"

show(p2)

Question 3: Monthly Box Plots
3.  Create box plots to visualize the distribution of temperatures for each month.

    * Extract the month from the 'Date' column and create a new 'Month' column.
    * Group the data by 'Month' and prepare it for plotting.
    * Use Bokeh's box plot elements to visualize the distribution.
    * Label the x-axis with month names and the y-axis with "Temperature (°C)".
    * Add tooltips to display the month and relevant statistical values (min, max, media

In [18]:
import numpy as np

month_names = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
               "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

df['Month'] = df['Date'].dt.month
df['MonthName'] = df['Date'].dt.month.apply(lambda x: month_names[x - 1])

grouped = df.groupby('MonthName')['Temperature']

stats = {
    'month': month_names,
    'q1':    [grouped.get_group(m).quantile(0.25) for m in month_names],
    'q2':    [grouped.get_group(m).quantile(0.50) for m in month_names],
    'q3':    [grouped.get_group(m).quantile(0.75) for m in month_names],
    'lower': [grouped.get_group(m).min()          for m in month_names],
    'upper': [grouped.get_group(m).max()           for m in month_names],
}

src3 = ColumnDataSource(stats)

hover3 = HoverTool(tooltips=[
    ("Month",  "@month"),
    ("Min",    "@lower °C"),
    ("Q1",     "@q1 °C"),
    ("Median", "@q2 °C"),
    ("Q3",     "@q3 °C"),
    ("Max",    "@upper °C"),
])

p3 = figure(
    title="Monthly Temperature Distribution",
    x_range=month_names,
    x_axis_label="Month",
    y_axis_label="Temperature (°C)",
    tools="pan,wheel_zoom,reset",
    width=900, height=450
)
p3.add_tools(hover3)

# Whiskers
p3.segment("month", "upper", "month", "q3", source=src3, line_color="black")
p3.segment("month", "lower", "month", "q1", source=src3, line_color="black")

# Boxes
p3.vbar("month", 0.7, "q2", "q3", source=src3, fill_color="lightblue", line_color="black")
p3.vbar("month", 0.7, "q1", "q2", source=src3, fill_color="steelblue",  line_color="black")

# Whisker caps
p3.rect("month", "upper", 0.3, 0.01, source=src3, line_color="black")
p3.rect("month", "lower", 0.3, 0.01, source=src3, line_color="black")

show(p3)

4.  Create box plots to visualize the distribution of temperatures for each year,
    and use color mapping to highlight temperature variations.

    * Extract the year from the 'Date' column and create a new 'Year' column.
    * Group the data by 'Year' and prepare it for plotting.
    * Use Bokeh's box plot elements to visualize the distribution for each year.
    * Label the x-axis with the 'Year' and the y-axis with "Temperature (°C)".
    * Use `factor_cmap` to color the boxes based on the median temperature of each year.
    * Add tooltips to display the year and relevant statistical values (min, max, median, etc.).
    * Enable pan, wheel zoom, and reset tools.

In [19]:
df['Year'] = df['Date'].dt.year.astype(str)
years = sorted(df['Year'].unique().tolist())

grouped_y = df.groupby('Year')['Temperature']

stats4 = {
    'year':  years,
    'q1':    [grouped_y.get_group(y).quantile(0.25) for y in years],
    'q2':    [grouped_y.get_group(y).quantile(0.50) for y in years],
    'q3':    [grouped_y.get_group(y).quantile(0.75) for y in years],
    'lower': [grouped_y.get_group(y).min()           for y in years],
    'upper': [grouped_y.get_group(y).max()            for y in years],
}

src4 = ColumnDataSource(stats4)

from bokeh.palettes import Blues9
palette = Blues9[:len(years)]

cmap = factor_cmap('year', palette=palette, factors=years)

hover4 = HoverTool(tooltips=[
    ("Year",   "@year"),
    ("Min",    "@lower °C"),
    ("Q1",     "@q1 °C"),
    ("Median", "@q2 °C"),
    ("Q3",     "@q3 °C"),
    ("Max",    "@upper °C"),
])

p4 = figure(
    title="Yearly Temperature Distribution",
    x_range=years,
    x_axis_label="Year",
    y_axis_label="Temperature (°C)",
    tools="pan,wheel_zoom,reset",
    width=900, height=450
)
p4.add_tools(hover4)

p4.segment("year", "upper", "year", "q3", source=src4, line_color="black")
p4.segment("year", "lower", "year", "q1", source=src4, line_color="black")

p4.vbar("year", 0.7, "q2", "q3", source=src4, fill_color=cmap, line_color="black")
p4.vbar("year", 0.7, "q1", "q2", source=src4, fill_color=cmap, line_color="black", fill_alpha=0.6)

p4.rect("year", "upper", 0.3, 0.01, source=src4, line_color="black")
p4.rect("year", "lower", 0.3, 0.01, source=src4, line_color="black")

show(p4)

Question 5: Interactive Time Range Selection

5.  Create an interactive line plot where the user can select a specific time range
    to view using a date range slider.

    * Create a basic line plot of 'Temperature' over 'Date'.
    * Implement a date range slider using Bokeh widgets to allow users to select a start and end date.
    * Update the plot dynamically based on the selected date range.
    * Add tooltips to display the date and temperature.
    * Enable pan, wheel zoom, and reset tools.

In [20]:
from bokeh.models import DateRangeSlider, CustomJS
from bokeh.layouts import column as bk_column

source_full = ColumnDataSource(df)
source_filtered = ColumnDataSource(df.copy())

hover5 = HoverTool(tooltips=[
    ("Date",        "@Date{%F}"),
    ("Temperature", "@Temperature °C")
], formatters={"@Date": "datetime"})

p5 = figure(
    title="Daily Minimum Temperatures (Interactive Range)",
    x_axis_label="Date",
    y_axis_label="Temperature (°C)",
    x_axis_type="datetime",
    tools="pan,wheel_zoom,reset",
    width=900, height=400
)
p5.add_tools(hover5)
p5.line("Date", "Temperature", source=source_filtered, line_width=1, color="steelblue")

slider = DateRangeSlider(
    title="Date Range",
    start=df['Date'].min(),
    end=df['Date'].max(),
    value=(df['Date'].min(), df['Date'].max()),
    step=1
)

callback = CustomJS(args=dict(source_full=source_full, source_filtered=source_filtered, slider=slider), code="""
    const [start, end] = slider.value;
    const full = source_full.data;
    const filtered = {Date: [], Temperature: []};
    
    for (let i = 0; i < full['Date'].length; i++) {
        if (full['Date'][i] >= start && full['Date'][i] <= end) {
            filtered['Date'].push(full['Date'][i]);
            filtered['Temperature'].push(full['Temperature'][i]);
        }
    }
    source_filtered.data = filtered;
    source_filtered.change.emit();
""")

slider.js_on_change('value', callback)

show(bk_column(slider, p5))

Question 6: Time Series Decomposition Visualization

6.  Perform a simple time series decomposition to visualize the trend and seasonality
    components of the temperature data.

    * Resample the data to monthly frequency and calculate the monthly average temperature.
    * Use a simple moving average to estimate the trend component.
    * Calculate the seasonal component by subtracting the trend from the original monthly data.
    * Create three separate Bokeh plots: one for the original monthly data, one for the trend,
        and one for the seasonal component.
    * Ensure the plots are aligned and share the same x-axis (Date).
    * Add tooltips to each plot to display the date and corresponding value.
    * Enable pan, wheel zoom, and reset tools for each plot.

In [21]:
# Rééchantillonnage mensuel
df_monthly = df.set_index('Date').resample('MS')['Temperature'].mean().reset_index()
df_monthly.columns = ['Date', 'Temperature']

# Tendance : moyenne mobile sur 12 mois
df_monthly['Trend'] = df_monthly['Temperature'].rolling(window=12, center=True).mean()

# Composante saisonnière
df_monthly['Seasonal'] = df_monthly['Temperature'] - df_monthly['Trend']

src6 = ColumnDataSource(df_monthly)

hover_orig = HoverTool(tooltips=[("Date", "@Date{%Y-%m}"), ("Temp", "@Temperature{0.2f} °C")],
                       formatters={"@Date": "datetime"})
hover_trend = HoverTool(tooltips=[("Date", "@Date{%Y-%m}"), ("Trend", "@Trend{0.2f} °C")],
                        formatters={"@Date": "datetime"})
hover_seas = HoverTool(tooltips=[("Date", "@Date{%Y-%m}"), ("Seasonal", "@Seasonal{0.2f} °C")],
                       formatters={"@Date": "datetime"})

kwargs = dict(x_axis_type="datetime", tools="pan,wheel_zoom,reset", width=900, height=280)

p_orig = figure(title="Monthly Average Temperature", **kwargs)
p_orig.add_tools(hover_orig)
p_orig.line("Date", "Temperature", source=src6, color="steelblue", line_width=2)

p_trend = figure(title="Trend (12-month moving average)", x_range=p_orig.x_range, **kwargs)
p_trend.add_tools(hover_trend)
p_trend.line("Date", "Trend", source=src6, color="firebrick", line_width=2)

p_seas = figure(title="Seasonal Component", x_range=p_orig.x_range, **kwargs)
p_seas.add_tools(hover_seas)
p_seas.line("Date", "Seasonal", source=src6, color="green", line_width=2)

show(bk_column(p_orig, p_trend, p_seas))